# S7b on Colab: test-split evaluation of the frozen student (the 20 "even" test tasks)

Runs `student epoch3` (adapter hash-checked against results/threshold.md) on 20 test tasks x 3 trials, in chunks of 4 tasks.
Each finished chunk is copied to Google Drive and skipped if you rerun after a disconnect.

**Before running**
1. Runtime > Change runtime type > **T4 GPU**.
2. Secrets (key icon in the left bar): add `HF_TOKEN` and `API_DEEPSEEK`, and switch **Notebook access** on for both.
3. In Google Drive > My Drive, a folder `agentic_distill_colab/` containing `agentic_distill_src.zip` and the folder `epoch3/`
   (with `adapter_config.json` and `adapter_model.safetensors`).
4. Start before Monday 06:30 IST (DeepSeek peak pricing; the runner refuses to start in peak hours).


In [ ]:
import sys, os, subprocess, json, hashlib, re, shutil, glob
print(sys.version)
assert sys.version_info >= (3, 12), "tau2 needs Python >= 3.12; stop and report"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
from google.colab import drive, userdata
drive.mount("/content/drive")
D = "/content/drive/MyDrive/agentic_distill_colab"
assert os.path.exists(f"{D}/agentic_distill_src.zip") and os.path.exists(f"{D}/epoch3/adapter_model.safetensors"), "upload the folder to Drive"
for k in ("HF_TOKEN", "API_DEEPSEEK"):
    os.environ[k] = userdata.get(k)                       # value never printed
print("secrets loaded:", [k for k in ("HF_TOKEN", "API_DEEPSEEK") if os.environ.get(k)])


In [ ]:
%%bash
set -e
rm -rf /content/repo && mkdir -p /content/repo
cd /content/repo && unzip -q /content/drive/MyDrive/agentic_distill_colab/agentic_distill_src.zip
ls results/threshold.md
grep -q disable_sdpa_gqa src/agentic_distill/student_run.py && grep -q "_p{os.getpid()}" src/agentic_distill/student_run.py
mkdir -p third_party && git clone -q https://github.com/sierra-research/tau2-bench third_party/tau2-bench
git -C third_party/tau2-bench checkout -q c06b07dd7c4d4ec61ba6ff9bcdf0b15dd7ef36d4
pip uninstall -y -q torchao || true
pip install -q -e third_party/tau2-bench audioop-lts pyyaml
pip install -q -U transformers peft accelerate
python -c "import tau2, transformers, peft, torch; print('ok; torch', torch.__version__, 'transformers', transformers.__version__, 'peft', peft.__version__)"


In [ ]:
# The checkpoint must be the one frozen in results/threshold.md. Copy it locally (faster than reading Drive each chunk).
os.makedirs("/content/epoch3", exist_ok=True)
for f in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy(f"{D}/epoch3/{f}", f"/content/epoch3/{f}")
frozen = re.search(r"sha256 `([0-9a-f]{64})`", open("/content/repo/results/threshold.md").read()).group(1)
h = hashlib.sha256(open("/content/epoch3/adapter_model.safetensors", "rb").read()).hexdigest()
print("adapter sha256:", h); assert h == frozen, "adapter differs from the one frozen in threshold.md"
import torch, transformers, peft   # record the environment for the write-up
os.makedirs(f"{D}/results", exist_ok=True)
json.dump({"python": sys.version, "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
           "gpu": torch.cuda.get_device_name(0)}, open(f"{D}/results/env_colab.json", "w"), indent=1)


In [ ]:
split = json.load(open("/content/repo/data/splits/retail_split.json"))
TEST = split["test_ids"]; assert len(TEST) == 40
EVEN = TEST[0::2]                                   # the half attempt 1 on Kaggle never ran
CHUNK = 4
chunks = [EVEN[i:i + CHUNK] for i in range(0, len(EVEN), CHUNK)]
print(len(EVEN), "tasks in", len(chunks), "chunks:", chunks)

def run_chunk(i, tasks):
    done = f"{D}/results/chunk{i}/DONE"
    if os.path.exists(done):
        print(f"chunk {i}: already done, skipping"); return
    out = f"/content/out_chunk{i}"; shutil.rmtree(out, ignore_errors=True)
    env = {**os.environ, "PYTHONPATH": "src", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    cmd = ["python", "-m", "agentic_distill.student_run", "--tasks", *tasks, "--split", "test", "--trials", "3",
           "--slice", "eval_student", "--label", "s2-epoch3", "--adapter", "/content/epoch3", "--out-root", out]
    print(f"chunk {i}: {tasks}", flush=True)
    p = subprocess.Popen(cmd, cwd="/content/repo", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if line.startswith(("task=", "slice", "STOP", "Traceback")) or "Error" in line[:40]:
            print(f"[chunk {i}] {line}", end="", flush=True)
    assert p.wait() == 0, f"chunk {i}: student_run exited {p.returncode}"
    dst = f"{D}/results/chunk{i}"; shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(f"{out}/data/raw", dst)
    open(done, "w").write("ok")
    print(f"chunk {i}: saved to Drive", flush=True)

for i, tasks in enumerate(chunks):
    run_chunk(i, tasks)


In [ ]:
recs = []
for f in sorted(glob.glob(f"{D}/results/chunk*/*/episodes.jsonl")):
    recs += [json.loads(l) for l in open(f)]
print(len(recs), "episode records (expect 60 when all chunks are done)")
print("success:", sum(r["reward"] > 0.5 for r in recs), "| crashes:", sum(r["terminated_by"] == "crash" for r in recs),
      "| aborts:", sum(r["terminated_by"] == "abort" for r in recs))
